# 🌊 Flood Detection & Multi-Class Segmentation
## AISEHack 2026 — Phase 2 | Theme 1 | IBM Collaboration

---

**Author:** Tanisha Sharma  
**HuggingFace Model:** [Tanisha0311/flood-segmentation-segformer-b5](https://huggingface.co/Tanisha0311/flood-segmentation-segformer-b5)  
**GitHub:** [ANRF AISEHack Phase-2](https://github.com/Tanisharma122/ANRF---AISEHack---Phase-2---Theme-1---Flood-Detection-IBM-)

---

## 📋 Table of Contents
1. [Project Overview](#overview)
2. [Problem Statement](#problem)
3. [Dataset Description](#dataset)
4. [Model Architecture](#architecture)
5. [Data Preprocessing](#preprocessing)
6. [Training Pipeline](#training)
7. [Results & Evaluation](#results)
8. [Visualizations](#viz)
9. [Inference & Deployment](#inference)
10. [Key Takeaways](#takeaways)

---
## 1. 🗺️ Project Overview <a id='overview'></a>

This project solves **multi-class flood segmentation** from **SAR (Synthetic Aperture Radar) satellite imagery** using a fine-tuned **SegFormer-B5** transformer model.

### Why SAR?
- SAR sensors work **day & night**, and **through cloud cover** — critical for real disaster scenarios
- Flood events frequently occur under overcast, rainy conditions where optical sensors fail
- SAR backscatter is highly sensitive to changes in surface water

### Phase 2 Advancement
- **Phase 1**: Binary classification (Flood / No Flood)
- **Phase 2**: Fine-grained 3-class segmentation (No Flood / Flood / Water Body)

| Class | ID | Description |
|-------|----|-------------|
| No Flood | 0 | Dry land, unaffected terrain |
| Flood | 1 | Actively inundated areas |
| Water Body | 2 | Permanent water (rivers, lakes, reservoirs) |

---
## 2. ❗ Problem Statement <a id='problem'></a>

Floods are among the world's most devastating natural disasters. **Rapid, accurate mapping** of flooded areas is critical for:
- Emergency response and evacuation routing
- Damage assessment and resource allocation
- Distinguishing newly flooded land from permanent water bodies

### Key Challenges
- **Class imbalance**: Flood pixels are far fewer than no-flood/water pixels
- **Similar visual appearance**: Flooded land and water bodies can look identical in SAR
- **Multi-band fusion**: Effectively combining multiple SAR frequency bands
- **Generalization**: Model must work across different geographies and flood events

---
## 3. 📁 Dataset Description <a id='dataset'></a>

The dataset consists of **multi-band SAR satellite imagery** tiles:
- Each input image contains **multiple spectral/polarization bands**
- Labels are per-pixel masks with 3 classes: `0` (No Flood), `1` (Flood), `2` (Water Body)
- Images are processed at **512×512 resolution**

### Data Split
- **Training set**: Used for model optimization
- **Validation set**: Track mIoU and Flood-class IoU during training
- **Holdout/Test set**: Final evaluation

In [ ]:
# ============================================================
# SECTION: Install & Import Dependencies
# ============================================================
# !pip install torch torchvision transformers numpy Pillow matplotlib scikit-learn

import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## 4. 🧠 Model Architecture <a id='architecture'></a>

### SegFormer-B5
We use **SegFormer-B5** — the largest and most powerful variant in the SegFormer family.

```
Input SAR Image (H × W × Bands)
         │
         ▼
┌─────────────────────────────┐
│   MiT-B5 Encoder            │
│   (Mix Transformer Blocks)  │
│   4 Hierarchical Stages     │
│   Multi-Scale Features      │
└────────────┬────────────────┘
             │  F1, F2, F3, F4  (multi-scale feature maps)
             ▼
┌─────────────────────────────┐
│   All-MLP Decoder Head      │
│   Fuses multi-scale features│
│   Lightweight & efficient   │
└────────────┬────────────────┘
             │
             ▼
   Segmentation Map (H × W × 3)
   Classes: [No Flood, Flood, Water Body]
```

### Why SegFormer?
- **No positional encoding** → works at variable resolutions
- **Efficient self-attention** via sequence reduction
- **Superior to CNNs** on segmentation benchmarks
- **B5 variant** gives maximum accuracy with strong generalization

In [ ]:
# ============================================================
# SECTION: Model Initialization
# ============================================================

NUM_CLASSES = 3  # 0: No Flood, 1: Flood, 2: Water Body
MODEL_NAME = "nvidia/mit-b5"

def build_model(num_classes=3, pretrained_model="nvidia/mit-b5"):
    """
    Build SegFormer-B5 model for multi-class flood segmentation.
    
    Args:
        num_classes: Number of segmentation classes
        pretrained_model: HuggingFace model ID for backbone
    Returns:
        Configured SegFormer model
    """
    model = SegformerForSemanticSegmentation.from_pretrained(
        pretrained_model,
        num_labels=num_classes,
        ignore_mismatched_sizes=True
    )
    return model

# Build the model
model = build_model(NUM_CLASSES)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

---
## 5. 🔧 Data Preprocessing <a id='preprocessing'></a>

### Normalization Strategy
- Per-band **mean** and **standard deviation** computed from the training set
- Saved as `.npy` files for consistent inference at deployment time
- Formula: `normalized = (pixel_value - mean) / std`

### Class Imbalance Handling
- Flood pixels are significantly underrepresented
- **Class weights** computed and applied in loss function
- Weights saved in `band_class_weights.npy`

In [ ]:
# ============================================================
# SECTION: Load Saved Normalization Statistics
# ============================================================

band_means = np.load('band_means.npy')
band_stds = np.load('band_stds.npy')
class_weights = np.load('band_class_weights.npy')

print("Band Means:", band_means)
print("Band Stds: ", band_stds)
print("Class Weights:", class_weights)
print("  → Class 0 (No Flood):   ", class_weights[0])
print("  → Class 1 (Flood):      ", class_weights[1])
print("  → Class 2 (Water Body): ", class_weights[2])

# Convert to tensor for loss function
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f"\nClass weights tensor on {device}: {class_weights_tensor}")

In [ ]:
# ============================================================
# SECTION: Custom Dataset Class
# ============================================================

class FloodSegmentationDataset(Dataset):
    """
    PyTorch Dataset for multi-class flood segmentation.
    
    Loads multi-band SAR images and corresponding segmentation masks.
    Applies per-band normalization using pre-computed statistics.
    """
    
    def __init__(self, image_paths, mask_paths, band_means, band_stds, image_size=512):
        """
        Args:
            image_paths: List of paths to SAR image files
            mask_paths:  List of paths to segmentation mask files
            band_means:  Per-band mean values for normalization
            band_stds:   Per-band std deviations for normalization
            image_size:  Target image size (square)
        """
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.band_means = band_means
        self.band_stds = band_stds
        self.image_size = image_size
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image (multi-band SAR)
        image = np.load(self.image_paths[idx]).astype(np.float32)  # (C, H, W)
        mask = np.load(self.mask_paths[idx]).astype(np.int64)      # (H, W)
        
        # Per-band normalization
        for c in range(image.shape[0]):
            image[c] = (image[c] - self.band_means[c]) / (self.band_stds[c] + 1e-8)
        
        return {
            'pixel_values': torch.tensor(image, dtype=torch.float32),
            'labels': torch.tensor(mask, dtype=torch.long)
        }

print("FloodSegmentationDataset class defined.")

---
## 6. 🏋️ Training Pipeline <a id='training'></a>

### Loss Function
Combined loss for robust training:
1. **Weighted Cross-Entropy Loss** — handles class imbalance
2. **Dice Loss** — maximizes region overlap, especially for minority flood class

```
Total Loss = CrossEntropy(class_weighted) + DiceLoss
```

### Optimizer & Schedule
- **Optimizer**: AdamW (weight decay for regularization)
- **LR Schedule**: CosineAnnealingLR for smooth convergence
- **Mixed Precision (AMP)**: Enabled for faster training on GPU

### Checkpointing Strategy
Two separate checkpoints saved:
- `best_model_mIoU.pth` — best model by **mean IoU** over all 3 classes
- `best_model_floodIoU.pth` — best model by **Flood-class IoU** specifically

In [ ]:
# ============================================================
# SECTION: Loss Functions
# ============================================================

class DiceLoss(nn.Module):
    """Dice Loss for segmentation — maximizes overlap between prediction and ground truth."""
    
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
    
    def forward(self, logits, targets, num_classes=3):
        probs = torch.softmax(logits, dim=1)
        dice_loss = 0.0
        
        for cls in range(num_classes):
            prob_cls = probs[:, cls]
            target_cls = (targets == cls).float()
            intersection = (prob_cls * target_cls).sum()
            dice = (2 * intersection + self.smooth) / (prob_cls.sum() + target_cls.sum() + self.smooth)
            dice_loss += (1 - dice)
        
        return dice_loss / num_classes


class CombinedLoss(nn.Module):
    """Combined Weighted Cross-Entropy + Dice Loss."""
    
    def __init__(self, class_weights, alpha=0.5):
        super().__init__()
        self.ce_loss = nn.CrossEntropyLoss(weight=class_weights)
        self.dice_loss = DiceLoss()
        self.alpha = alpha  # Weight between CE and Dice
    
    def forward(self, logits, targets):
        ce = self.ce_loss(logits, targets)
        dice = self.dice_loss(logits, targets)
        return self.alpha * ce + (1 - self.alpha) * dice


criterion = CombinedLoss(class_weights_tensor)
print("Loss functions initialized.")

In [ ]:
# ============================================================
# SECTION: Evaluation Metrics — IoU / mIoU
# ============================================================

def compute_iou(pred_mask, true_mask, num_classes=3):
    """
    Compute per-class Intersection over Union (IoU) and mean IoU.
    
    Args:
        pred_mask: Predicted class labels (H, W)
        true_mask: Ground truth class labels (H, W)
        num_classes: Number of segmentation classes
    
    Returns:
        per_class_iou: List of IoU per class
        mean_iou: Mean IoU across all classes
    """
    per_class_iou = []
    
    for cls in range(num_classes):
        pred_cls = (pred_mask == cls)
        true_cls = (true_mask == cls)
        
        intersection = (pred_cls & true_cls).sum().item()
        union = (pred_cls | true_cls).sum().item()
        
        iou = intersection / (union + 1e-8)
        per_class_iou.append(iou)
    
    mean_iou = np.mean(per_class_iou)
    return per_class_iou, mean_iou

print("IoU metric function defined.")
print("Classes tracked: 0=No Flood | 1=Flood | 2=Water Body")

In [ ]:
# ============================================================
# SECTION: Training Loop (Illustrative)
# ============================================================
# Note: This shows the training structure used. Actual training
# was run in the full experiment notebook on GPU.

def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device):
    """
    Train model for one epoch with mixed precision (AMP).
    
    Returns:
        avg_loss: Average training loss for the epoch
        avg_miou: Average mIoU across the training set
    """
    model.train()
    total_loss = 0.0
    total_miou = 0.0
    
    for batch in dataloader:
        pixel_values = batch['pixel_values'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision forward pass
        with torch.cuda.amp.autocast():
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits  # (B, num_classes, H/4, W/4)
            
            # Upsample to original resolution
            upsampled = nn.functional.interpolate(
                logits, size=labels.shape[-2:], mode='bilinear', align_corners=False
            )
            loss = criterion(upsampled, labels)
        
        # Mixed precision backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        # Compute IoU
        preds = upsampled.argmax(dim=1)
        _, miou = compute_iou(preds.cpu(), labels.cpu())
        
        total_loss += loss.item()
        total_miou += miou
    
    n = len(dataloader)
    return total_loss / n, total_miou / n


def validate(model, dataloader, criterion, device):
    """Validate model and compute mIoU and per-class IoU."""
    model.eval()
    total_loss = 0.0
    all_class_ious = []
    
    with torch.no_grad():
        for batch in dataloader:
            pixel_values = batch['pixel_values'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(pixel_values=pixel_values)
            logits = outputs.logits
            upsampled = nn.functional.interpolate(
                logits, size=labels.shape[-2:], mode='bilinear', align_corners=False
            )
            loss = criterion(upsampled, labels)
            total_loss += loss.item()
            
            preds = upsampled.argmax(dim=1)
            per_class_iou, _ = compute_iou(preds.cpu(), labels.cpu())
            all_class_ious.append(per_class_iou)
    
    avg_iou_per_class = np.mean(all_class_ious, axis=0)
    miou = np.mean(avg_iou_per_class)
    flood_iou = avg_iou_per_class[1]  # Class 1 = Flood
    
    return total_loss / len(dataloader), miou, flood_iou, avg_iou_per_class

print("Training and validation functions defined.")

---
## 7. 📊 Results & Evaluation <a id='results'></a>

### Model Checkpoints

| Checkpoint | Optimized For | Description |
|------------|---------------|-------------|
| `best_model_mIoU.pth` | Mean IoU | Best overall segmentation performance across all 3 classes |
| `best_model_floodIoU.pth` | Flood IoU | Best performance specifically on the critical Flood class |

### Why Two Checkpoints?
In disaster detection, **recall on the flood class** is more critical than overall accuracy. A model that is conservative about flood class may have high mIoU but miss actual flood pixels. The `best_model_floodIoU.pth` prioritizes this use case.

---
## 8. 📈 Visualizations <a id='viz'></a>

In [ ]:
# ============================================================
# SECTION: Training Curves Visualization
# ============================================================

plt.figure(figsize=(14, 5))
img = plt.imread('training_curves.png')
plt.imshow(img)
plt.axis('off')
plt.title('Training & Validation Curves — Loss and IoU Over Epochs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
print("The training curves show model convergence over training epochs.")

In [ ]:
# ============================================================
# SECTION: Validation Predictions Visualization
# ============================================================

plt.figure(figsize=(16, 10))
img = plt.imread('val_predictions.png')
plt.imshow(img)
plt.axis('off')
plt.title('Validation Set Predictions: SAR Input | Ground Truth | Predicted Mask', 
          fontsize=14, fontweight='bold')

# Add legend
legend_elements = [
    mpatches.Patch(facecolor='black', label='Class 0: No Flood'),
    mpatches.Patch(facecolor='red',   label='Class 1: Flood'),
    mpatches.Patch(facecolor='blue',  label='Class 2: Water Body'),
]
plt.legend(handles=legend_elements, loc='lower right', fontsize=11)
plt.tight_layout()
plt.show()

---
## 9. 🚀 Inference & Deployment <a id='inference'></a>

### Option A: Load from HuggingFace (Recommended)
The model is publicly available on HuggingFace: [Tanisha0311/flood-segmentation-segformer-b5](https://huggingface.co/Tanisha0311/flood-segmentation-segformer-b5)

In [ ]:
# ============================================================
# SECTION: Inference — Load from HuggingFace
# ============================================================

from transformers import SegformerForSemanticSegmentation
import torch

HF_MODEL_ID = "Tanisha0311/flood-segmentation-segformer-b5"

# Load model from HuggingFace Hub
hf_model = SegformerForSemanticSegmentation.from_pretrained(
    HF_MODEL_ID,
    num_labels=3,
    ignore_mismatched_sizes=True
)
hf_model.eval()
print(f"✅ Model loaded from HuggingFace: {HF_MODEL_ID}")

In [ ]:
# ============================================================
# SECTION: Inference — Load Local Checkpoints
# ============================================================

from transformers import SegformerForSemanticSegmentation
import torch
import torch.nn.functional as F
import numpy as np

# ---- Choose checkpoint ----
# best_model_mIoU.pth      → best overall (all 3 classes)
# best_model_floodIoU.pth  → best for flood detection specifically

CHECKPOINT_PATH = 'best_model_mIoU.pth'  # or 'best_model_floodIoU.pth'

# Build model and load weights
local_model = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/mit-b5",
    num_labels=3,
    ignore_mismatched_sizes=True
)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu')
local_model.load_state_dict(checkpoint)
local_model.eval()
print(f"✅ Loaded checkpoint: {CHECKPOINT_PATH}")


def predict_segmentation(model, image_array, band_means, band_stds):
    """
    Run segmentation inference on a single SAR image.
    
    Args:
        model: Loaded SegFormer model
        image_array: SAR image as numpy array (C, H, W)
        band_means: Per-band normalization means
        band_stds:  Per-band normalization stds
    
    Returns:
        pred_mask: Predicted segmentation mask (H, W) with class labels
    """
    # Normalize
    img = image_array.astype(np.float32).copy()
    for c in range(img.shape[0]):
        img[c] = (img[c] - band_means[c]) / (band_stds[c] + 1e-8)
    
    # Add batch dimension
    tensor = torch.tensor(img).unsqueeze(0)  # (1, C, H, W)
    
    with torch.no_grad():
        outputs = model(pixel_values=tensor)
        logits = outputs.logits  # (1, num_classes, H/4, W/4)
        
        # Upsample to original resolution
        upsampled = F.interpolate(
            logits, size=image_array.shape[-2:], 
            mode='bilinear', align_corners=False
        )
        pred_mask = upsampled.argmax(dim=1).squeeze(0).numpy()
    
    return pred_mask  # Values: 0=No Flood, 1=Flood, 2=Water Body

print("Inference pipeline ready.")
print("Call predict_segmentation(model, sar_image, band_means, band_stds)")

---
## 10. 🔑 Key Takeaways <a id='takeaways'></a>

| # | Takeaway |
|---|----------|
| 1 | **SegFormer-B5** achieves superior segmentation through hierarchical transformer encoding |
| 2 | **SAR imagery** enables flood detection in all weather conditions, crucial for real disasters |
| 3 | **Combined CE + Dice loss** effectively handles both class imbalance and overlap optimization |
| 4 | **Two checkpoints** (mIoU & Flood IoU) serve different operational needs |
| 5 | **Saved normalization stats** (means/stds) are essential for consistent, reproducible inference |
| 6 | **Mixed precision (AMP)** significantly reduces training time without sacrificing accuracy |
| 7 | Model is **publicly hosted on HuggingFace** for easy zero-setup deployment |
| 8 | Distinguishing **Flood vs Water Body** is a key innovation over binary Phase 1 approach |

---

## 🔗 Resources

- 🤗 **HuggingFace Model**: [Tanisha0311/flood-segmentation-segformer-b5](https://huggingface.co/Tanisha0311/flood-segmentation-segformer-b5)
- 🐙 **GitHub Repository**: [ANRF AISEHack Phase-2](https://github.com/Tanisharma122/ANRF---AISEHack---Phase-2---Theme-1---Flood-Detection-IBM-)
- 📄 **SegFormer Paper**: [SegFormer: Simple and Efficient Design for Semantic Segmentation with Transformers](https://arxiv.org/abs/2105.15203)

---
*Built with ❤️ for disaster resilience | AISEHack 2026 | IBM*